# Scientific figure exports

Regenerate the accepted mechanical trade-off scatter, individual response curves and
initial-beam displacement comparison from an external figure manifest. This notebook
reads saved physical arrays, CSV properties and the declared specimen INPs. It does
not load a checkpoint, run inference, train, or export new simulation data.

Run from the configured repository environment so `resources` is importable. Set the
manifest and output controls below. Approved figure-library exports remain unchanged;
new figures default to the saved run's `results/postProcessing/figures/` directory.

In [ ]:
from pathlib import Path

FIGURE_MANIFEST = None  # Explicit path to the external figure-manifest.json.
OUTPUT_ROOT = None      # None: saved-run results/postProcessing/figures.
FORMATS = ("png", "svg")  # Add "pdf" when a vector PDF export is wanted.
DPI = 400
VERIFY_SOURCES = True

## Sources and output contract

The manifest fixes the saved configuration, selected specimens, source hashes,
physical units, masks, frame and reported aggregate metrics. Hash verification is
on by default; an unavailable or changed declared source fails before it is plotted.
The current model-winner document is not a regeneration dependency.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from resources.damage import fcc_map
from resources.MLmetrics import (
    plot_mechanical_tradeoffs,
    plot_curve_comparison_panels,
    prepare_beam_plot_geometry,
    plot_lattice_field_comparison,
)

if FIGURE_MANIFEST is None:
    raise ValueError("Set FIGURE_MANIFEST to the external figure-manifest.json.")
manifest_path = Path(FIGURE_MANIFEST).expanduser().resolve()
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
field_spec = manifest["fields"]
predictions_path = Path(field_spec["source_predictions"])
OUTPUT_ROOT = (Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT is not None
               else predictions_path.parent / "postProcessing" / "figures")
library_root = manifest_path.parent.resolve()
if OUTPUT_ROOT == library_root or library_root in OUTPUT_ROOT.parents:
    raise ValueError("Choose a separate OUTPUT_ROOT; approved library files are immutable here.")
FORMATS = tuple(str(fmt).lower() for fmt in FORMATS)
if not FORMATS or set(FORMATS) - {"png", "svg", "pdf"} or DPI <= 0:
    raise ValueError("Choose png/svg/pdf formats and a positive DPI.")
source_checks, generated_files = {}, {}


def verify_source(path, expected_sha256):
    path = Path(path)
    digest = None
    if VERIFY_SOURCES:
        hasher = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                hasher.update(block)
        digest = hasher.hexdigest()
        if digest != expected_sha256:
            raise ValueError(f"Source hash differs from the manifest: {path}")
    source_checks[str(path)] = {"expected_sha256": expected_sha256,
                                "verified_sha256": digest}
    return path


def save_figure(fig, prefix):
    stem = OUTPUT_ROOT / prefix
    stem.parent.mkdir(parents=True, exist_ok=True)
    try:
        for fmt in FORMATS:
            path = stem.with_suffix("." + fmt)
            fig.savefig(path, dpi=DPI, facecolor="white", transparent=False)
            generated_files[str(path)] = {"sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                                          "bytes": path.stat().st_size}
    finally:
        plt.close(fig)


verify_source(predictions_path, field_spec["source_predictions_sha256"])
print(f"Regeneration output: {OUTPUT_ROOT}")

## Mechanical trade-offs

Every disordered design remains an individual point. Properties are normalised by
the declared periodic reference; colour is the same design's tensile strength.
These display limits and ticks reproduce the accepted scatter without aggregating data.

In [ ]:
scatter_spec = manifest["scatter"]
properties_path = verify_source(scatter_spec["source_csv"], scatter_spec["source_csv_sha256"])
properties = pd.read_csv(properties_path, index_col=0)
properties.index = pd.to_numeric(properties.index, errors="raise").astype(int)

SCATTER_STYLE = {
    "xlim": (0.88, 1.85), "ylim": (0.68, 1.27),
    "xticks": (1.0, 1.2, 1.4, 1.6, 1.8),
    "yticks": (0.8, 1.0, 1.2), "color_ticks": (0.92, 0.94, 0.96, 0.98),
}
fig, ax, colorbar = plot_mechanical_tradeoffs(
    properties,
    periodic_id=scatter_spec["periodic_reference_id"],
    highlight_id=scatter_spec["circled_specimen_id"],
    figsize_mm=scatter_spec["dimensions_mm"], font_size=25,
    point_size=scatter_spec["marker_area_pt2"], **SCATTER_STYLE,
)
save_figure(fig, scatter_spec["files_prefix"])

## Individual response predictions

Use the manifest's fixed favourable examples, not population means. Simulation,
Prediction and transparent absolute pointwise residual bars share the response axis.
The fracture force arrays are saved in N and are converted to kN for display.

In [ ]:
curve_spec = manifest["curves"]
TASK_STYLE = {"UT": {"label": "Tension", "color": "#007F86"},
              "FT": {"label": "Fracture", "color": "#C4454E"}}
curve_panels = []
with np.load(predictions_path, allow_pickle=False) as arrays:
    sample_ids = arrays["sample_ids"].astype(int)
    if len(np.unique(sample_ids)) != len(sample_ids):
        raise ValueError("Saved sample IDs must be unique.")
    for task, case in curve_spec["cases"].items():
        selected = np.flatnonzero(sample_ids == case["specimen_id"])
        if len(selected) != 1:
            raise ValueError(f"Curve specimen missing or duplicated: {task}, {case['specimen_id']}")
        index = int(selected[0])
        force_scale = 1000.0 if task == "FT" else 1.0
        truth = arrays[f"{task}_val_curve_truth"][index].astype(float) / force_scale
        prediction = arrays[f"{task}_val_curve_outputs"][index].astype(float) / force_scale
        x = arrays[f"{task}_curve_x_values"].astype(float).reshape(-1)
        curve_panels.append({
            "x": x, "truth": truth, "prediction": prediction, **TASK_STYLE[task],
            "xlabel": "Displacement (mm)" if task == "FT" else "Strain",
            "ylabel": "Force (kN)" if task == "FT" else "Stress (MPa)",
        })
fig, axes = plot_curve_comparison_panels(
    curve_panels, figsize_mm=curve_spec["dimensions_mm"], font_size=25,
)
save_figure(fig, curve_spec["files_prefix"])

## Displacement on verified initial beams

Dimensions come from the saved canonical coordinates, frame values and component
labels. The declared U2 slice uses each task's presence and saved validity masks.
`fcc_map` checks the selected INP; the adapter aligns its native beam chains to the
saved ML node order. Empty cells and the initial crack stay empty. Colour is a
linear display interpolation of saved endpoint displacement on the initial geometry,
not evolving damage or a deformed mesh. The reported overall RMSE is supplied by
the manifest and describes the complete validation prediction, not this selected slice.

In [ ]:
FIELD_VIEW = {
    "UT": {"xlim": (-4, 204), "ylim": (-4, 194), "rmse_format": ".2f"},
    "FT": {"xlim": tuple(field_spec["FT_zoom_mm"]["x"]),
           "ylim": tuple(field_spec["FT_zoom_mm"]["y"]), "rmse_format": ".3f"},
}
field_panels = []
with np.load(predictions_path, allow_pickle=False) as arrays:
    sample_ids = arrays["sample_ids"].astype(int)
    canonical_coords = arrays["canonical_coords"].astype(float)
    n_nodes = len(canonical_coords)
    for task, case in field_spec["cases"].items():
        selected = np.flatnonzero(sample_ids == case["specimen_id"])
        if len(selected) != 1:
            raise ValueError(f"Field specimen missing or duplicated: {task}, {case['specimen_id']}")
        index = int(selected[0])
        frames = arrays[f"{task}_frame_values"]
        components = arrays[f"{task}_components"].astype(str).tolist()
        n_frames, n_components = len(frames), len(components)
        frame = case["retained_frame_one_based"] - 1
        component = 1
        if not 0 <= frame < n_frames or len(components) <= component or components[component] != "U2":
            raise ValueError("The declared frame and saved U2 component must be present.")
        expected_shape = (n_nodes, n_frames * n_components)
        truth = arrays[f"{task}_val_field_truth"][index].astype(float)
        prediction = arrays[f"{task}_val_field_outputs"][index].astype(float)
        saved_mask = arrays[f"{task}_val_field_mask"][index].astype(bool)
        if truth.shape != expected_shape or prediction.shape != expected_shape or saved_mask.shape != expected_shape:
            raise ValueError("Saved node/frame/component dimensions disagree.")
        truth = truth.reshape(n_nodes, n_frames, n_components)[:, frame, component]
        prediction = prediction.reshape(n_nodes, n_frames, n_components)[:, frame, component]
        saved_mask = saved_mask.reshape(n_nodes, n_frames, n_components)[:, frame, component]
        node_mask = arrays[f"{task}_node_mask"].astype(bool)
        valid_mask = node_mask & saved_mask & np.isfinite(truth) & np.isfinite(prediction)
        inp = verify_source(case["inp_path"], case["inp_sha256"])
        beam_map, _, _ = fcc_map(inp, task)
        geometry = prepare_beam_plot_geometry(beam_map, canonical_coords, node_mask)
        if len(geometry["edges"]) != case["actual_initial_struts"]:
            raise ValueError("Verified INP strut count differs from the figure manifest.")
        field_panels.append({
            "truth": truth, "prediction": prediction, "valid_mask": valid_mask,
            **geometry, **TASK_STYLE[task], **FIELD_VIEW[task],
            "beam_width": case["physical_in_plane_width_mm"],
            "value_limit": max(abs(v) for v in case["truth_prediction_limits_mm"]),
            "error_limit": max(abs(v) for v in case["signed_difference_limits_mm"]),
            "overall_rmse": field_spec["overall_RMSE_mm"][task], "units": "mm",
        })
fig, axes = plot_lattice_field_comparison(
    field_panels, figsize_mm=field_spec["dimensions_mm"], font_size=28, metric_font_size=25,
)
save_figure(fig, field_spec["files_prefix"])

## Regeneration record

The output record binds these new exports to their manifest and read-only inputs.
It does not promote a new model winner, reselect examples, or alter approved assets.

In [ ]:
regeneration = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "manifest": str(manifest_path),
    "manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    "source_verification_enabled": VERIFY_SOURCES,
    "sources": source_checks,
    "outputs": generated_files,
    "curve_cases": {task: case["specimen_id"] for task, case in curve_spec["cases"].items()},
    "field_cases": {task: {"specimen_id": case["specimen_id"],
                           "frame_one_based": case["retained_frame_one_based"]}
                    for task, case in field_spec["cases"].items()},
}
report_path = OUTPUT_ROOT / "figure-regeneration.json"
report_path.write_text(json.dumps(regeneration, indent=2), encoding="utf-8")
print(f"Wrote {len(generated_files)} figure files and {report_path}")